In [1]:
!pip install -q transformers peft accelerate bitsandbytes safetensors torch pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch, re, csv, os, math
import xml.etree.ElementTree as ET
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
ADAPTER_PATH = '/kaggle/input/notebooks/rebeccachenn/model-done/fast/'

print('Files in adapter dir:', os.listdir(ADAPTER_PATH))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# ✅ SPEED: left-padding enables batched generation
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH, is_trainable=False)
model.eval()
model.config.use_cache = True
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()

print('Loaded! Device:', next(model.parameters()).device)


Files in adapter dir: ['adapter_model.safetensors', 'checkpoint-2000', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'checkpoint-1000']


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded! Device: cuda:0


In [4]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
DISALLOWED_ATTRS_RE = re.compile(
    r'\s+(?:filling|onclick|onload|onmouseover|xmlns:xlink)=["\'][^"\']*("|\')',
    re.IGNORECASE
)
SVG_REGEX = re.compile(r'(<svg\b[\s\S]*?</svg>)', flags=re.IGNORECASE)

def normalize_svg(svg_text):
    if not svg_text: return ''
    svg_text = DISALLOWED_ATTRS_RE.sub('', svg_text)
    for attr in ('width', 'height', 'viewBox'):
        svg_text = re.sub(rf'(<svg\b[^>]*?)\b{attr}=["\'][^"\']*("|\')', r'\1', svg_text)
    svg_text = re.sub(
        r'(<svg\b)',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"',
        svg_text, count=1
    )
    svg_text = re.sub(r'(xmlns="[^"]*")(?=.*xmlns="[^"]*")', '', svg_text)
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text

def has_disallowed_tags(svg_text):
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            tag = elem.tag
            if isinstance(tag, str):
                local = tag.split('}')[-1] if '}' in tag else tag
                if local not in ALLOWED_TAGS: return True
        return False
    except Exception: return True

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        return tag == 'svg'
    except ET.ParseError: return False

def extract_svg(text):
    if not text: return ''
    for tok in ['<|im_end|>', '<|endoftext|>']:
        idx = text.find(tok)
        if idx != -1: text = text[:idx]
    match = SVG_REGEX.search(text)
    if match: return match.group(1).strip()
    start = text.lower().find('<svg')
    if start != -1:
        partial = text[start:].strip()
        if not partial.lower().endswith('</svg>'): partial += '</svg>'
        return partial
    return ''

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

print('Helpers defined.')


Helpers defined.


In [5]:
SYSTEM_PROMPT = (
    'You are an SVG code generator. '
    'Given a text description, output ONLY valid SVG code with '
    'width="256" height="256" viewBox="0 0 256 256". '
    'Do not include explanations or markdown code fences.'
)

test_df = pd.read_csv(
    '/kaggle/input/datasets/rebeccachenn/dlmidterm/test.csv',
    engine='python', quoting=csv.QUOTE_MINIMAL
)
print('test_df shape:', test_df.shape)
test_df.head(2)


test_df shape: (1000, 2)


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...


## Batched Inference

Key speed improvements:
- **Batch size = 4**: process 4 prompts per forward pass (~3x faster than 1-by-1)
- **max_new_tokens = 350**: most SVGs finish before 500 tokens
- **OOM guard**: auto-falls back to single inference if GPU runs out of memory
- **Partial save every 100 rows**: safe to resume if session crashes

In [7]:
BATCH_SIZE = 4        # ✅ Process 4 prompts at once — ~3x faster than 1-by-1
MAX_NEW_TOKENS = 350  # ✅ Most SVGs finish < 350 tokens; raise if fallback rate is high
SAVE_DIR = '/kaggle/working/'
os.makedirs(SAVE_DIR, exist_ok=True)

def make_prompt(prompt_text):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt_text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(prompts: list, max_new_tokens=MAX_NEW_TOKENS):
    """
    Generate SVGs for a list of prompts in one forward pass.
    Returns list of (svg, used_fallback) tuples.
    """
    chat_texts = [make_prompt(p) for p in prompts]

    # ✅ left-padding was set at load time — required for batched generation
    inputs = tokenizer(
        chat_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,      # cap input length to avoid OOM
    ).to(model.device)

    with torch.no_grad():
        with torch.cuda.amp.autocast():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

    results = []
    input_len = inputs['input_ids'].shape[1]  # same for all (left-padded)
    for i in range(len(prompts)):
        decoded = tokenizer.decode(output_ids[i][input_len:], skip_special_tokens=False)
        svg = normalize_svg(extract_svg(decoded))
        used_fallback = not is_valid_svg(svg) or has_disallowed_tags(svg) or len(svg) > 15900
        if used_fallback:
            svg = fallback_svg()
        results.append((svg, used_fallback))
    return results


# ── Main inference loop ──────────────────────────────────────────────
all_results = []
fallback_count = 0
n = len(test_df)

for batch_start in range(0, n, BATCH_SIZE):
    batch_rows = test_df.iloc[batch_start : batch_start + BATCH_SIZE]
    prompts = batch_rows['prompt'].tolist()
    ids     = batch_rows['id'].tolist()

    try:
        batch_results = generate_batch(prompts)
    except RuntimeError as e:
        # OOM fallback: process one-by-one
        print(f'  OOM at batch {batch_start}, falling back to single inference')
        torch.cuda.empty_cache()
        batch_results = []
        for p in prompts:
            try:
                res = generate_batch([p])
                batch_results.extend(res)
            except Exception:
                batch_results.append((fallback_svg(), True))

    for row_id, (svg, fb) in zip(ids, batch_results):
        if fb: fallback_count += 1
        all_results.append({'id': row_id, 'svg': svg})

    processed = min(batch_start + BATCH_SIZE, n)
    if processed % 100 == 0 or processed == n:
        print(f'Processed {processed}/{n} | fallback: {fallback_count}')
        part_num = processed // 100
        partial_df = pd.DataFrame(all_results)
        partial_path = os.path.join(SAVE_DIR, f'submission_partial_{part_num}.csv')
        partial_df.to_csv(partial_path, index=False)
        print(f'Saved checkpoint: {partial_path}')

submission_df = pd.DataFrame(all_results)
print(f'\nDone! Total:{len(submission_df)} | Fallback:{fallback_count} ({fallback_count/len(submission_df)*100:.1f}%)')


/tmp/ipykernel_55/1229337609.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Processed 100/1000 | fallback: 74
Saved checkpoint: /kaggle/working/submission_partial_1.csv
Processed 200/1000 | fallback: 142
Saved checkpoint: /kaggle/working/submission_partial_2.csv
Processed 300/1000 | fallback: 219
Saved checkpoint: /kaggle/working/submission_partial_3.csv
Processed 400/1000 | fallback: 295
Saved checkpoint: /kaggle/working/submission_partial_4.csv
Processed 500/1000 | fallback: 371
Saved checkpoint: /kaggle/working/submission_partial_5.csv
Processed 600/1000 | fallback: 442
Saved checkpoint: /kaggle/working/submission_partial_6.csv
Processed 700/1000 | fallback: 517
Saved checkpoint: /kaggle/working/submission_partial_7.csv
Processed 800/1000 | fallback: 597
Saved checkpoint: /kaggle/working/submission_partial_8.csv
Processed 900/1000 | fallback: 672
Saved checkpoint: /kaggle/working/submission_partial_9.csv
Processed 1000/1000 | fallback: 747
Saved checkpoint: /kaggle/working/submission_partial_10.csv

Done! Total:1000 | Fallback:747 (74.7%)


In [8]:
valid_count = submission_df['svg'].apply(is_valid_svg).sum()
print(f'Valid          : {valid_count}/{len(submission_df)} ({valid_count/len(submission_df)*100:.1f}%)')
print(f'Disallowed tags: {submission_df["svg"].apply(has_disallowed_tags).sum()}')
print(f'Over 16000 chars: {(submission_df["svg"].str.len() > 16000).sum()}')

final_path = os.path.join(SAVE_DIR, 'submission.csv')
submission_df.to_csv(final_path, index=False)
print('Saved:', final_path)
submission_df.head()


Valid          : 1000/1000 (100.0%)
Disallowed tags: 0
Over 16000 chars: 0
Saved: /kaggle/working/submission.csv


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg width=""256"" height=""256"" viewBox=""0 0 256..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
